In [1]:
from simulator import Simulator
from simulator.config import DATA_PATH, Color
from simulator.helpers import clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
              QGC,
              Gazebo,
              GazMarker,
              NoVisualizer,
              QGCMarker,
              SimVehicle,
)

clean()

In [2]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=90) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 

base_home = ENUPose(x=5, y=5, z=0, heading=45)
base_path = Plan.create_square_path(side_len=10, alt=5)

In [3]:
enu_home = enu_origin.to_abs(base_home)
gra_home = gra_origin.to_abs(base_home)
enu_waypoints = enu_home.to_abs_all(base_path) 
gra_waypoints = gra_home.to_abs_all(base_path)

In [4]:
sysid = 1 
color = Color.BLUE

In [5]:
mission_path = DATA_PATH / f"mission_{sysid}.waypoints"
auto_plan = AutoPlan(name="simple_auto_plan", mission_path=str(mission_path))
auto_plan.save_basic_mission(sysid=sysid,gra_wps=GRAPose.unpose_all(gra_waypoints))

In [6]:
guided_plan = GuidedPlan(
              name="simple_guided_plan",
              wps=ENUPose.unpose_all(enu_waypoints),
)

In [7]:
veh = SimVehicle(sysid=sysid,
              gcs_name='Simple',
              plan=guided_plan,
              color = color,
              home=enu_home,
              waypoints=ENUPose.unpose_all(enu_waypoints))

In [8]:
gaz= Gazebo(gra_origin,world_path="simulator/gazebo/worlds/runway.world")
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

In [9]:
qgc= QGC(gra_origin)
origin_qgc = QGCMarker(name="origin",
                pos=gra_origin.unpose(),
                color=Color.WHITE)
qgc.markers.append(origin_qgc)

In [10]:
novis = NoVisualizer(gra_origin)

In [11]:
simulator = Simulator(
	visualizer=novis,
	terminals=['gcs'],
	verbose=1,
)

simulator.add_vehicle(veh)

simulator.show()

[NovisVehicle(home=ENUPose(x=-5.0, y=5.0, z=0, heading=135))]


In [12]:
orac = simulator.launch()

00:29:46 - Oracle ⚪ - INFO - 🙈 Running without visualization.
00:29:46 - Oracle ⚪ - INFO - 🚀 GCS Simple launched (PID 4962)


/home/ubuntu/uav-cyber-sim/simulator/data


In [13]:
orac.run()

00:29:46 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 1 vehicles and 1 GCSs
# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
# Failed to load module "canberra-gtk-module"
# Failed to load module "canberra-gtk-module"


KeyboardInterrupt: 

In [ ]:
import threading
for thread in threading.enumerate():
    print(f"{thread.name}: alive={thread.is_alive()}, daemon={thread.daemon}")